In [48]:
import json
import pandas as pd

In [49]:
with open('../data/raw/data-original.json',"r") as file:
    tree=json.load(file)

In [50]:
nodes=pd.DataFrame(tree["nodes"])

In [51]:
nodes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 338530 entries, 0 to 338529
Data columns (total 9 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   id        338530 non-null  int64  
 1   name      338524 non-null  object 
 2   thesis    307398 non-null  object 
 3   school    281804 non-null  object 
 4   country   322984 non-null  object 
 5   year      277239 non-null  float64
 6   subject   202490 non-null  object 
 7   advisors  338530 non-null  object 
 8   students  338530 non-null  object 
dtypes: float64(1), int64(1), object(7)
memory usage: 23.2+ MB


In [52]:
nodes.students

0           [11, 28, 31]
1         [19930, 19939]
2                     []
3                     []
4                     []
               ...      
338525                []
338526                []
338527                []
338528                []
338529                []
Name: students, Length: 338530, dtype: object

In [53]:
nodes.students.dtype

dtype('O')

In [54]:
new_nodes = pd.DataFrame(tree["nodes"], index = nodes.index)

In [55]:
new_nodes.head(10)

,id,name,thesis,school,country,year,subject,advisors,students
0,1,Ernest Willard Anderson,Statics of Special Types of Homogeneous Elasti...,Iowa State University,UnitedStates,1933.0,74—Mechanics of deformable solids,[258],"[11, 28, 31]"
1,4,Charles Joseph Thorne,The Approximate Solution of Linear Differentia...,Iowa State University,UnitedStates,1941.0,None,[239],"[19930, 19939]"
2,5,Ralph Harry Tripp,Statical Equilibrium of Skew and Sector-Shaped...,Iowa State University,UnitedStates,1942.0,None,[258],[]
3,2,Archie Higdon,Stresses in Moderately Thick Rectangular Plates,Iowa State University,UnitedStates,1936.0,74—Mechanics of deformable solids,[258],[]
4,3,Donald Hill Rock,Finite Strain Analysis in Elastic Theory,Iowa State University,UnitedStates,1939.0,None,[258],[]
5,6,William B. Stiles,Solutions of Clamped Plated Problems by Means ...,Iowa State University,UnitedStates,1945.0,None,[258],[]
6,9,Henry David Block,Explicit Solutions of Certain Singular Integra...,Iowa State University,UnitedStates,1949.0,None,[281],"[23, 28, 71, 101996, 87826]"
7,8,James W. Beach,Flow of Viscous Fluid between Slowing Rotating...,Iowa State University,UnitedStates,1948.0,None,[258],[]
8,7,Carl Eric Langenhop,Properties of Kernels of Integral Equations Wh...,Iowa State University,UnitedStates,1948.0,None,[281],"[48, 104689, 38, 33, 3026, 39, 32, 35]"
9,10,Frank E. Bortle,Analytical Study of Dynamic Loads on Elastical...,Iowa State University,UnitedStates,1949.0,None,[258],[]


In [ ]:
from collections import defaultdict
import ast

def dataframe_to_graph_from_students(df, id_col="id", students_col="students"):
    graph = defaultdict(list)

    for _, row in df.iterrows():
        parent = int(row[id_col])
        students = row[students_col]

        # If students are stored as strings like "[11, 28, 31]",
        # convert them to actual Python lists.
        if isinstance(students, str):
            students = ast.literal_eval(students)

        # If students is missing or empty
        if students is None:
            students = []

        for child in students:
            child = int(child)
            graph[parent].append(child)

            # Make sure the child also appears as a node
            graph.setdefault(child, [])

        # Make sure parent appears even if it has no students
        graph.setdefault(parent, [])

    return dict(graph)

In [57]:
graph = dataframe_to_graph_from_students(new_nodes)

In [58]:
graph

{1: [11, 28, 31],
 11: [],
 28: [],
 31: [],
 4: [19930, 19939],
 19930: [],
 19939: [],
 5: [],
 2: [],
 3: [],
 6: [],
 9: [23, 28, 71, 101996, 87826],
 23: [],
 71: [],
 101996: [205728,
  105376,
  105370,
  105373,
  105374,
  105367,
  105336,
  105365,
  105368,
  105372,
  105369,
  105375,
  105366,
  284480,
  78224,
  105371,
  340769],
 87826: [],
 8: [],
 7: [48, 104689, 38, 33, 3026, 39, 32, 35],
 48: [124, 177, 3346, 173, 3331, 151, 185, 3667, 3673],
 104689: [],
 38: [12351, 12352, 3731, 12365, 12307, 3705, 3706],
 33: [],
 3026: [],
 39: [],
 32: [108682],
 35: [],
 10: [],
 12: [],
 13: [59464,
  59467,
  59466,
  59461,
  52007,
  16021,
  40989,
  12372,
  59465,
  265,
  59468,
  281193,
  13372,
  59463,
  59469,
  59462],
 59464: [],
 59467: [],
 59466: [118776],
 59461: [],
 52007: [],
 16021: [60688,
  79646,
  5980,
  148811,
  48367,
  12372,
  39499,
  149812,
  15022,
  96379,
  47622,
  39500,
  36942,
  36166,
  97722,
  11500,
  66910,
  77795,
  41929,


In [ ]:
def count_descendants(graph, start):
    visited = set()

    #recursive function call
    def dfs(node):
        for child in graph.get(node, []):
            if child not in visited:
                visited.add(child)
                dfs(child)

    dfs(start)
    return len(visited)

In [46]:
print(count_descendants(graph, 93643))

188
